In [6]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split

In [7]:
data = pd.read_csv("Iris.csv")
data = data.dropna(axis=1, how="all")

X = data.drop(columns=['Id', 'Species']).to_numpy()
y = data['Species'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

In [8]:
class Node:
    def __init__(self, feature_idx=None, threshold=None, info_gain=None, left=None, right=None, value=None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.info_gain = info_gain
        self.left = left
        self.right = right

        self.value = value

In [9]:
class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=2):
        self.root = None
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth

    def build_tree(self, dataset, curr_depth=0):
        X, y = dataset[:, :-1], dataset[:, -1]
        n_samples, n_features = X.shape

        if n_samples >= self.min_samples_split and curr_depth <= self.max_depth:
            best_split = self.best_split(dataset, n_features)

            if best_split["info_gain"] > 0:
                left_node = self.build_tree(best_split["left_datasets"], curr_depth + 1)
                right_node = self.build_tree(best_split["right_datasets"], curr_depth + 1)
                return Node(
                    feature_idx=best_split["feature_idx"],
                    threshold=best_split["threshold"],
                    left=left_node,
                    right=right_node,
                    info_gain=best_split["info_gain"]
                )

        leaf_value = self.calculate_leaf_value(y)
        return Node(value=leaf_value)

    def best_split(self, dataset, n_features):
        best_split = {"info_gain": -1}
        max_info_gain = -float("inf")

        for feature_idx in range(n_features):
            feature_values = dataset[:, feature_idx]
            possible_thresholds = np.unique(feature_values)

            for threshold in possible_thresholds:
                left_datasets, right_datasets = self.split(dataset, feature_idx, threshold)

                if len(left_datasets) > 0 and len(right_datasets) > 0:
                    y = dataset[:, -1]
                    left_y = left_datasets[:, -1]
                    right_y = right_datasets[:, -1]

                    curr_info_gain = self.information_gain(y, left_y, right_y)

                    if curr_info_gain > max_info_gain:
                        best_split["feature_idx"] = feature_idx
                        best_split["threshold"] = threshold
                        best_split["left_datasets"] = left_datasets
                        best_split["right_datasets"] = right_datasets
                        best_split["info_gain"] = curr_info_gain
                        max_info_gain = curr_info_gain

        return best_split

    def split(self, dataset, feature_idx, threshold):
        left_datasets = dataset[dataset[:, feature_idx] <= threshold]
        right_datasets = dataset[dataset[:, feature_idx] > threshold]
        return left_datasets, right_datasets

    def information_gain(self, parent, left_child, right_child):
        weight_l = len(left_child) / len(parent)
        weight_r = len(right_child) / len(parent)
        gain = self.entropy(parent) - (weight_l * self.entropy(left_child) + weight_r * self.entropy(right_child))
        return gain

    def entropy(self, y):
        class_labels = np.unique(y)
        entropy = 0
        for cls in class_labels:
            p_cls = len(y[y == cls]) / len(y)
            entropy += -p_cls * np.log2(p_cls + 1e-9)
        return entropy

    def calculate_leaf_value(self, y):
        y = list(y)
        return max(y, key=y.count)

    def fit(self, X, y):
        dataset = np.concatenate([X, y.reshape(-1, 1)], axis=1)
        self.root = self.build_tree(dataset)

    def predict(self, X):
        return np.array([self.make_prediction(x, self.root) for x in X])

    def make_prediction(self, x, tree):
        if tree.value is not None:
            return tree.value
        feature_val = x[tree.feature_idx]
        if feature_val <= tree.threshold:
            return self.make_prediction(x, tree.left)
        else:
            return self.make_prediction(x, tree.right)


In [10]:
dt = DecisionTree(min_samples_split=2, max_depth=2)
dt.fit(X_train, y_train)
predictions = dt.predict(X_test)

accuracy = np.mean(predictions == y_test) * 100
print(f'Accuracy: {accuracy:.2f}%')



Accuracy: 96.67%
